# S7.1 · 隐私攻击与防护评估

攻击者设定为**诚实但好奇的协议内参与方**，不假设外部窃听。

| 编号 | 攻击 | 谁攻击谁 | 暴露面 |
|---|---|---|---|
| A1 | 标签推断 | 被动方 → 主动方标签 | 每轮下发的残差 |
| A2 | 嵌入反演 | 主动方 → 被动方特征 | 上传的嵌入 |
| A3 | 梯度标签推断 | 被动方 → 主动方标签 | 回传的梯度（仅形态A） |

In [1]:
ROUND_DP = 4          # 表格展示精度（不影响任何计算结果）
import sys, subprocess, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "registry").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, yaml
CONFIG_PATH = ROOT / "modules/m5_modeling/configs/experiment.yaml"
config = yaml.safe_load(open(CONFIG_PATH, encoding="utf-8"))
seed = config.get("seeds", [config.get("seed")])[0]
git = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, cwd=ROOT).stdout.strip()
print("config:", CONFIG_PATH.relative_to(ROOT))
print("seed  :", seed, "| 全部种子:", config.get("seeds"))
print("git   :", git or "(未提交)")
print("numpy :", np.__version__, "| pandas:", pd.__version__)

config: modules/m5_modeling/configs/experiment.yaml
seed  : 11 | 全部种子: [11, 22, 33, 44, 55]
git   : 9dbefc0
numpy : 2.3.5 | pandas: 2.3.3


In [2]:
atk = pd.read_csv(ROOT / 'modules/m7_security/results/attack_results.csv')
a1 = atk[atk.attack == 'A1_残差标签推断'].groupby('dp_sigma')[
    ['eps_per_round','utility_auc','leak_auc_首轮','leak_auc_最优轮']].mean()
a1.round(ROUND_DP)

,eps_per_round,utility_auc,leak_auc_首轮,leak_auc_最优轮
dp_sigma,,,,
0.00,inf,0.7868,1.0000,1.0000
0.01,484.4805,0.7869,1.0000,1.0000
0.03,161.4935,0.7869,1.0000,1.0000
0.10,48.4481,0.7870,1.0000,1.0000
0.30,16.1494,0.7872,0.9878,0.9952
1.00,4.8448,0.7869,0.7574,0.7981
3.00,1.6149,0.7771,0.5933,0.6481
10.00,0.4845,0.7109,0.5291,0.5892


首轮泄露 AUC = 1.0000：训练开始时权重为零、预测恒为 0.5，残差 `r = 0.5 − y` 的符号与标签**一一对应**。**不加防护的纵向联邦逻辑回归，标签是完全泄露的。**

## 证伪检验：跨轮平均攻击

上表看起来 σ=1.0 就能把泄露压到 0.76 而几乎不损失可用性——这个结论**太好了**。

但攻击者只用了单轮残差。噪声在轮间独立、标签恒定，**跨轮平均即可把噪声消掉**。这是必须自己打的证伪。

In [3]:
mr = pd.read_csv(ROOT / 'modules/m7_security/results/multiround_attack.csv')
mr.groupby('dp_sigma')[['可用性AUC','单轮攻击','跨轮平均攻击','前50轮平均']].mean().round(ROUND_DP)

,可用性AUC,单轮攻击,跨轮平均攻击,前50轮平均
dp_sigma,,,,
0.0,0.7868,1.0000,1.0000,1.0000
0.3,0.7872,0.9878,1.0000,1.0000
1.0,0.7869,0.7574,1.0000,1.0000
3.0,0.7771,0.5933,0.9999,0.9446
10.0,0.7109,0.5291,0.9037,0.6851
30.0,0.6352,0.5149,0.6659,0.5659


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "Heiti TC", "PingFang SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
FIGSIZE_WIDE = (9, 4)
FIG_DPI = 150
GRID_W = 11
GRID_H = 7
FIGDIR = ROOT / "modules/m7_security" / "results"
m = mr.groupby('dp_sigma')[['可用性AUC','单轮攻击','跨轮平均攻击']].mean()
L0_REF = 0.7089
fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
ax.plot(m.index, m['单轮攻击'], marker='o', label='泄露 AUC（单轮攻击）')
ax.plot(m.index, m['跨轮平均攻击'], marker='s', label='泄露 AUC（跨轮平均攻击）')
ax.plot(m.index, m['可用性AUC'], marker='^', label='模型可用性 AUC')
ax.axhline(L0_REF, color='red', linestyle='--', label='L0 内地单方基线')
ax.set_xscale('symlog'); ax.set_xlabel('高斯噪声 σ'); ax.set_ylabel('AUC')
ax.set_title('逐轮加噪防护：跨轮平均攻击下失效')
ax.legend(); fig.tight_layout()
fig.savefig(FIGDIR / 'dp_privacy_utility.png', dpi=FIG_DPI); plt.close(fig)
print('图已保存 dp_privacy_utility.png')

图已保存 dp_privacy_utility.png


**结论（推翻了上一节的乐观读数）**：

- σ=1.0 时跨轮平均攻击的泄露 AUC 回到 **1.0000**
- 要把泄露压到 0.67，需要 σ=30，此时可用性降至 0.6352，**低于 L0 内地单方基线 0.7089**

→ **逐轮加高斯噪声不是本协议的有效防护**。把泄露压下去所需的噪声，会先把联邦模型的价值清零。有效路径只能是**协议级**手段（安全聚合 / 同态加密 / 秘密分享）或改变暴露面。

## A2 嵌入反演 / A3 梯度标签推断

In [5]:
a2 = atk[atk.attack == 'A2_嵌入反演'].groupby('protocol')[
    ['utility_auc','inv_r2_mean','inv_r2_max','leak_auc_梯度方向']].mean()
a2.round(ROUND_DP)

,utility_auc,inv_r2_mean,inv_r2_max,leak_auc_梯度方向
protocol,,,,
L3c_形态A_双向,0.7690,0.4877,0.7497,1.0
L3c_形态B_自监督,0.7402,0.6666,0.8745,0.5
L3c_形态B_随机,0.7267,0.4964,0.7592,0.5


## S7.4 · 恶意参与方：放开半诚实假设

此前全部结论的前提是**半诚实**——参与方严格执行协议，只是好奇。但这个假设在真实场景中往往不成立：协议由代码执行，而代码由某一方控制，对方无从验证收到的数字是不是真按协议算的。

**A6 恶意主动方**：不发真实残差，改发构造的探针向量。被动方按协议更新 `w_b -= lr·(x_bᵀ·r)/n`，若 r 是第 j 个样本上的单位向量，更新量正比于 `x_b[j]`——下一轮上传量之差直接给出 `x_b·x_b[j]`。有 d 个辅助样本，每个目标只需 **1 次探针**即可精确反解。

In [6]:
mp = pd.read_csv(ROOT / 'modules/m7_security/results/malicious_probe.csv')
t = mp.pivot_table(index='amplitude', columns='n_repeat',
                   values='feat_r2', aggfunc='mean')
t.columns = [f'重复×{c:,}' for c in t.columns]
t.round(ROUND_DP)

,重复×1,重复×100,"重复×10,000","重复×1,000,000","重复×100,000,000"
amplitude,,,,,
1.0,-1.0,-1.0000,-1.0000,-1.0000,0.9503
10.0,-1.0,-1.0000,-1.0000,0.9503,0.9995
1000.0,-1.0,0.9503,0.9995,1.0000,1.0000


**关键发现：上行加噪与残差合法性检查必须同时开启，缺一不可。**

| 防护组合 | 攻破 σ=0.1 所需成本 |
|---|---|
| 只加噪（幅度不受约束） | 幅度 1000 时**仅需 100 次重复** |
| 只查合法性（无噪声） | 伪装探针 R²=0.859，**26 次全部未被标记** |
| **两者同时** | 合法幅度下需约 **10⁸ 次重复**，实际不可行 |

机制：单次探针的信号量级是 `lr·amplitude/n`，噪声是固定的 σ——**信噪比与幅度成正比**。约束幅度的唯一手段就是合法性检查（真实残差 `r = sigmoid(logit) − y` 必落在 [−1, 1] 且稠密）。

→ 合法性检查把攻击成本抬高了**六个数量级**。它不是锦上添花，**它是让加噪防护有意义的前提**。

### A7 恶意被动方：定向抬分

这不是偷数据，是**操纵决策**。被动方送的部分 logit 直接加进最终打分，主动方无从验证它是不是真由 `x_b·w_b` 算出的。

In [7]:
mb = pd.read_csv(ROOT / 'modules/m7_security/results/malicious_boost.csv')
g = mb.groupby('amplitude')[['baseline_in_topk', 'attacked_in_topk',
                             'target_size', 'list_churn']].mean()
g['目标进入率'] = g.attacked_in_topk / g.target_size
g['名单重合度'] = 1 - g.list_churn
g[['baseline_in_topk', 'attacked_in_topk', '目标进入率', '名单重合度']].round(ROUND_DP)

,baseline_in_topk,attacked_in_topk,目标进入率,名单重合度
amplitude,,,,
0.0,14.0,14.0,0.0974,1.0000
0.5,14.0,35.4,0.2462,0.9704
1.0,14.0,64.8,0.4506,0.9297
2.0,14.0,120.8,0.8401,0.8520
5.0,14.0,143.8,1.0000,0.8201


**旧的名单稳定性阈值 0.9 抓不到中等幅度的操纵**：幅度 1.0 时 45% 的目标客户被顶进 Top-10% 名单，而名单重合度仍有 0.93 —— **高于 0.9，不会告警**。

→ M8 的防护基线已据此收紧到 **0.95**。但要说清楚：这只是**检测**手段，不是防护——攻击者压低幅度仍可缓慢渗透。根治需要主动方能验证被动方送来的部分 logit 确由 `x_b·w_b` 算出（零知识证明类手段），本阶段未实现。

两个反直觉的结果：

1. **形态B 挡住了梯度标签泄露**（A3 从 1.00 降到 0.50）——不回传梯度，暴露面直接消失。
2. **形态B 并没有降低特征反演风险，反而更糟**（自监督形态 R²=0.667 > 形态A 的 0.488）。PCA 编码器是线性的、更容易求逆；随标签训练的编码器反而丢掉了更多与任务无关的信息。

→ **「冻结编码器 = 更安全」是错的**。它换掉的是标签暴露面，不是特征暴露面。

## A4 特征推断：本项目最严重的单项发现

威胁模型：主动方每轮收到被动方上传的部分 logit `x_b · w_b`，并另有**少量样本**的 x_b 真值（辅助集）。攻击分两步最小二乘：

1. 用辅助集解出每轮的 w_b（需辅助样本数 ≥ 特征维数）；
2. 用解出的 w_b 反解其余所有样本的 x_b。

In [8]:
fi = pd.read_csv(ROOT / 'modules/m7_security/results/feature_inference.csv')
fi[fi.uplink_sigma == 0].groupby('uplink_sigma')[
    ['可用性AUC','特征R²均值','特征R²最差维','条件数']].mean().round(ROUND_DP)

,可用性AUC,特征R²均值,特征R²最差维,条件数
uplink_sigma,,,,
0.0,0.7868,1.0,1.0,68647.0787


**无防护时特征被精确恢复（R² = 1.0000）**。加上 A1 的标签完全泄露，**无防护的纵向联邦逻辑回归是双向完全泄露的**——「原始数据不出本地」在这个协议下不构成任何实质保护。

### 上行加噪：与标签推断完全相反的结果

被动方对**上传的部分 logit**加噪（`uplink_sigma`），这与 A1 的下行加噪是不同参与方的自我保护，不可互相替代。

In [9]:
ud = pd.read_csv(ROOT / 'modules/m7_security/results/uplink_defense_utility.csv')
fi2 = fi.groupby('uplink_sigma')[['特征R²均值']].mean()
ud2 = ud.groupby('uplink_sigma')[['训练与推理均加噪','L0']].mean()
ud2.join(fi2).round(ROUND_DP)

,训练与推理均加噪,L0,特征R²均值
uplink_sigma,,,
0.00,0.7868,0.7089,1.0000
0.01,0.7870,0.7089,0.4669
0.03,0.7858,0.7089,0.3538
0.10,0.7854,0.7089,0.1964
0.30,0.7684,0.7089,0.0715
1.00,0.7205,0.7089,0.0277
3.00,0.6192,0.7089,NaN


**结论与 A1 相反：上行加噪是有效防护，且代价极小。**

- σ=0.1：特征 R² 从 1.0000 降到 0.196，可用性仅损失 0.0014
- σ=1.0：R² 降到 0.028，但可用性损失 0.066（VFL 增益的 85%）

机制解释：A1 的信号（标签）**轮间恒定**，跨轮平均即可消噪；A4 需要解一个**病态线性系统**（条件数约 4.8×10⁴），噪声被放大五个数量级。**同一种手段对两类攻击效果相反，因此防护方案必须逐攻击面评估，不能一刀切。**

### 证伪检验：更强的攻击能否绕过？

In [10]:
st = pd.read_csv(ROOT / 'modules/m7_security/results/feature_inference_stronger.csv')
st.groupby(['uplink_sigma','n_aux'])[['朴素最小二乘','加强版(最优岭系数)']].mean().round(ROUND_DP)

朴素最小二乘  加强版(最优岭系数)
uplink_sigma n_aux                    
0.00         8      1.0000      1.0000
             64     1.0000      1.0000
             512    1.0000      1.0000
             2000   1.0000      1.0000
0.01         8      0.2872      0.3423
             64     0.4673      0.4813
             512    0.2165      0.5148
             2000  -0.7225      0.5546
0.10         8      0.0690      0.1613
             64     0.1900      0.2165
             512   -0.3676      0.2256
             2000  -2.3132      0.2798
1.00         8      0.0617      0.1043
             64     0.0231      0.1273
             512   -0.8716      0.1290
             2000  -3.6043      0.1292

加强版攻击（岭正则化 + 更大辅助集）确实优于朴素最小二乘，但**没有推翻结论**：σ=1.0 时即使给攻击者 2000 个辅助样本（已知训练集大部分 x_b 真值，这是极其宽松的假设），R² 也只到 0.129。

注意朴素最小二乘在噪声下**辅助样本越多反而越差**（σ=1.0、n_aux=2000 时 R² = −3.60）——只报朴素版本会在相反方向上误导读者，故两版并列报告。

## A5 成员推断：先验证攻击，再下结论

「这个人是否在你的建模样本里」本身就是个人信息，即便特征与标签都没泄露。

**但朴素的损失阈值攻击没有通过有效性验证**：在刻意制造记忆的模型上（训练 AUC = 1.0000、过拟合间隙 0.32），它也只能达到 0.52。攻击弱到这个程度，就**不能据此宣称「成员推断无威胁」**。

In [11]:
mc = pd.read_csv(ROOT / 'modules/m7_security/results/membership_capacity.csv')
g = mc.groupby(['depth','rounds'])[['训练AUC','测试AUC','membership_auc']].mean()
g['过拟合间隙'] = g['训练AUC'] - g['测试AUC']
g.round(ROUND_DP)

,,训练AUC,测试AUC,membership_auc,过拟合间隙
depth,rounds,,,,
3,60,0.9822,0.6926,0.5155,0.2896
6,60,1.0000,0.6850,0.5146,0.3150
8,200,1.0000,0.6839,0.5205,0.3161
10,400,1.0000,0.6840,0.5238,0.3160
12,800,1.0000,0.6906,0.5226,0.3094


因此改用 **LiRA 式影子模型校准**：逐样本比较「在训练集内」与「不在训练集内」两种情形下的置信度分布。它随模型容量单调增强，**通过了有效性验证**。

In [12]:
lira = pd.read_csv(ROOT / 'modules/m7_security/results/membership_lira.csv')
lira.groupby('level').membership_auc_lira.agg(['mean','std']).round(ROUND_DP)

,mean,std
level,,
L0_内地单方,0.5105,0.0062
L1_加k匿名统计,0.5105,0.0068
L3b_纵向GBDT_深3,0.5277,0.0092
L3b_纵向GBDT_深6,0.5453,0.0070
L4_集中式/L3信息集,0.5103,0.0022


**结论：成员泄露由模型容量驱动，与是否联邦无关。**

- L0 内地单方（0.5105）≈ L1（0.5105）≈ L4/L3 信息集（0.5103）——引入对方数据**不增加**成员泄露
- 纵向 GBDT 深 3（0.528）→ 深 6（0.545）——**容量才是驱动因素**

→ 工程含义：控制树深，而不是纠结要不要联邦。

> **降级说明**：本实现用 32 个影子模型，LiRA 原文通常用 64–256；且本项目样本量约 2000、特征维数低，模型本身记忆有限。因此上述数值应读作**该设定下的下界**，不构成「成员推断不可行」的一般结论。